# 03 - Verification Evaluation

Notebook đánh giá bước **claim verification** cho bài toán Vietnamese Historical Fact Verification.

File này dùng đúng schema dữ liệu hiện tại:

- `dev_dataset.jsonl` / `test_dataset.jsonl`: claim + gold label + gold evidence.
- `hybrid_reranker_top5_all_labels_dev.jsonl` / `hybrid_reranker_top5_all_labels_test.jsonl`: claim + top-5 `retrieved_evidence`.

Ba setting đánh giá:

1. `claim_only`: chỉ đưa claim vào LLM.
2. `gold_evidence`: đưa claim + gold evidence từ dataset.
3. `retrieved_evidence`: đưa claim + top-5 retrieved evidence từ retrieval/reranker.

Notebook này không đọc corpus, không truy vấn Qdrant, không chạy retrieval. Nó chỉ đọc các file JSONL đã có.

In [1]:
# # # !pip -q uninstall -y transformers accelerate bitsandbytes tokenizers huggingface-hub safetensors

!pip -q install --no-cache-dir \
  "numpy==2.0.2" \
  "pandas==2.2.2" \
  "scikit-learn>=1.6.1" \
  "tqdm>=4.66,<5.0" \
  "safetensors==0.5.3" \
  "huggingface-hub==0.30.2" \
  "tokenizers==0.21.1" \
  "transformers==4.51.3" \
  "accelerate==1.6.0" \
  "bitsandbytes==0.48.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 190.3 MB/s eta 0:00:00a 0:00:01


In [2]:
import sys
import gc
import re
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import importlib.metadata as md
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

for package_name in ["transformers", "accelerate", "bitsandbytes", "scikit-learn", "pandas", "numpy", "tqdm", "sentencepiece"]:
    try:
        print(f"{package_name}: {md.version(package_name)}")
    except Exception:
        print(f"{package_name}: not installed")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA: 12.8
CUDA available: True
GPU count: 2
0 Tesla T4
1 Tesla T4
transformers: 4.51.3
accelerate: 1.6.0
bitsandbytes: 0.48.2
scikit-learn: 1.6.1
pandas: 2.2.2
numpy: 2.0.2
tqdm: 4.67.3
sentencepiece: 0.2.1


In [3]:
# Chỉ đổi biến này khi chuyển từ dev sang test.
RUN_TAG = "dev"  # "dev" hoặc "test"

DATASET_PATHS = {
    "dev": "/kaggle/input/datasets/lanthanhvi/dataset/dev_dataset.jsonl",
    "test": "/kaggle/input/datasets/lanthanhvi/dataset/test_dataset.jsonl",
}

RETRIEVED_EVIDENCE_PATHS = {
    "dev": "/kaggle/input/datasets/lanthanhvi/retrieval-output-dev/hybrid_reranker_alpha05_top5_all_labels_dev.jsonl",
    "test": "/kaggle/input/datasets/lanthanhvi/retrieval-output-dev/hybrid_alpha05_top5_all_labels_dev.jsonl",
}

# Dùng thư mục output mới để tránh lẫn với notebook cũ.
OUTPUT_DIR = Path("/kaggle/working/datasets/lanthanhvi/verification_outputs_clean")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    {"tag": "qwen25_3b_8bit", "name": "Qwen/Qwen2.5-3B-Instruct", "quantization": "8bit"},
    {"tag": "qwen25_7b_8bit", "name": "Qwen/Qwen2.5-7B-Instruct", "quantization": "8bit"},
]

SETTINGS = ["claim_only", "gold_evidence", "retrieved_evidence"]

LABELS = ["SUPPORTED", "REFUTED", "NOT_ENOUGH_EVIDENCE"]
EXTENDED_LABELS = LABELS + ["PARSE_ERROR"]

MAX_SAMPLES = None       # Đặt số nhỏ, ví dụ 10, nếu muốn test nhanh.
MAX_NEW_TOKENS = 512
MAX_EVIDENCE_ITEMS = 5
FORCE_RERUN = True      # True nếu muốn chạy lại dù output đã tồn tại.

In [4]:
def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() != ".jsonl":
        raise ValueError(f"Expected .jsonl file, got: {path}")

    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON at {path}, line {line_no}") from exc
            if not isinstance(obj, dict):
                raise ValueError(f"Each JSONL line must be an object. Error at {path}, line {line_no}")
            rows.append(obj)
    return rows


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def require_fields(row, required_fields, row_name):
    missing = [field for field in required_fields if field not in row]
    if missing:
        raise ValueError(f"Missing fields in {row_name}: {missing}")


def validate_dataset(rows):
    required = ["claim_id", "chunk_id", "pages", "raw_text", "claim", "label", "claim_type", "evidence_ids", "explanation", "split"]
    seen = set()

    for idx, row in enumerate(rows):
        row_name = f"dataset row {idx}"
        require_fields(row, required, row_name)

        claim_id = str(row["claim_id"])
        if claim_id in seen:
            raise ValueError(f"Duplicate claim_id in dataset: {claim_id}")
        seen.add(claim_id)

        if row["label"] not in LABELS:
            raise ValueError(f"Invalid label at {row_name}: {row['label']}")
        if not isinstance(row["evidence_ids"], list) or len(row["evidence_ids"]) == 0:
            raise ValueError(f"evidence_ids must be a non-empty list at {row_name}")
        if not isinstance(row["raw_text"], str) or not row["raw_text"].strip():
            raise ValueError(f"raw_text must be a non-empty string at {row_name}")
        if not isinstance(row["claim"], str) or not row["claim"].strip():
            raise ValueError(f"claim must be a non-empty string at {row_name}")

    return True


def validate_retrieval_rows(rows):
    required_top = ["claim_id", "retrieved_evidence"]
    required_ev = ["rank", "chunk_id", "pages", "raw_text"]
    seen = set()

    for idx, row in enumerate(rows):
        row_name = f"retrieval row {idx}"
        require_fields(row, required_top, row_name)

        claim_id = str(row["claim_id"])
        if claim_id in seen:
            raise ValueError(f"Duplicate claim_id in retrieval file: {claim_id}")
        seen.add(claim_id)

        evidences = row["retrieved_evidence"]
        if not isinstance(evidences, list) or len(evidences) == 0:
            raise ValueError(f"retrieved_evidence must be a non-empty list at {row_name}")

        for ev_idx, ev in enumerate(evidences):
            ev_name = f"retrieval row {idx}, evidence {ev_idx}"
            if not isinstance(ev, dict):
                raise ValueError(f"Evidence must be an object at {ev_name}")
            require_fields(ev, required_ev, ev_name)
            if not isinstance(ev["raw_text"], str) or not ev["raw_text"].strip():
                raise ValueError(f"raw_text must be a non-empty string at {ev_name}")

    return True


def build_retrieved_map(rows):
    # Schema cố định: mỗi row có claim_id và retrieved_evidence.
    return {str(row["claim_id"]): row["retrieved_evidence"] for row in rows}


def validate_dataset_retrieval_match(dataset_rows, retrieved_map):
    dataset_ids = {str(row["claim_id"]) for row in dataset_rows}
    retrieval_ids = set(retrieved_map.keys())

    missing = sorted(dataset_ids - retrieval_ids)
    extra = sorted(retrieval_ids - dataset_ids)

    if missing:
        preview = missing[:10]
        raise ValueError(f"Missing retrieved_evidence for {len(missing)} claim(s). Examples: {preview}")

    if extra:
        print(f"Warning: retrieval file has {len(extra)} extra claim_id(s) not used by dataset.")

    return True

In [5]:
dataset = load_jsonl(DATASET_PATHS[RUN_TAG])
if MAX_SAMPLES is not None:
    dataset = dataset[:MAX_SAMPLES]

retrieval_rows = load_jsonl(RETRIEVED_EVIDENCE_PATHS[RUN_TAG])

validate_dataset(dataset)
validate_retrieval_rows(retrieval_rows)
retrieved_map = build_retrieved_map(retrieval_rows)
validate_dataset_retrieval_match(dataset, retrieved_map)

print("RUN_TAG:", RUN_TAG)
print("Dataset size:", len(dataset))
print("Retrieved map size:", len(retrieved_map))
print("\nLabel distribution:")
print(pd.Series([row["label"] for row in dataset]).value_counts())
print("\nClaim type distribution:")
print(pd.Series([row["claim_type"] for row in dataset]).value_counts())

RUN_TAG: dev
Dataset size: 264
Retrieved map size: 264

Label distribution:
SUPPORTED              88
REFUTED                88
NOT_ENOUGH_EVIDENCE    88
Name: count, dtype: int64

Claim type distribution:
MILITARY_DIPLOMACY          101
PERSON_ROLE                  52
TIME_EVENT                   42
POLITICAL_ADMINISTRATIVE     35
GENEALOGY_SUCCESSION         22
NATURAL_OMEN_DISASTER        12
Name: count, dtype: int64


##  Format evidence và tạo prompt

- Gold evidence: `item["raw_text"]`, `item["pages"]`, `item["evidence_ids"]`.
- Retrieved evidence: `ev["rank"]`, `ev["chunk_id"]`, `ev["pages"]`, `ev["raw_text"]`.

## Parse output và gọi LLM

In [6]:
def format_gold_evidence(item):
    pages = item["pages"]
    raw_text = item["raw_text"].strip()
    return f"[Gold evidence: pages={pages}]\n{raw_text}"


def get_retrieval_info(item):
    claim_id = str(item["claim_id"])
    evidences = retrieved_map[claim_id][:MAX_EVIDENCE_ITEMS]

    gold_evidence_ids = {str(x) for x in item["evidence_ids"]}
    retrieved_chunk_ids = [str(ev["chunk_id"]) for ev in evidences]

    gold_rank = None
    for ev in evidences:
        ev_chunk_id = str(ev["chunk_id"])
        if ev_chunk_id in gold_evidence_ids:
            try:
                gold_rank = int(ev["rank"])
            except Exception:
                gold_rank = ev["rank"]
            break

    retrieval_hit = gold_rank is not None
    missing_retrieval = len(evidences) == 0

    return {
        "evidences": evidences,
        "gold_evidence_ids": sorted(gold_evidence_ids),
        "retrieved_chunk_ids": retrieved_chunk_ids,
        "retrieval_hit": retrieval_hit,
        "gold_rank": gold_rank,
        "missing_retrieval": missing_retrieval,
    }


def format_retrieved_evidence(item):
    info = get_retrieval_info(item)
    blocks = []

    for ev in info["evidences"]:
        rank = ev["rank"]
        pages = ev["pages"]
        raw_text = ev["raw_text"].strip()
        blocks.append(f"[Evidence {rank}: pages={pages}]\n{raw_text}")

    return "\n\n".join(blocks), info


def build_prompt(item, setting):
    claim = item["claim"].strip()

    if setting == "claim_only":
        return f"""You are a Vietnamese historical claim verifier.

Determine whether the following historical claim is correct based on your knowledge.

Labels:
- SUPPORTED: The claim is historically correct.
- REFUTED: The claim contains a factual error. The error may involve a person, role, title, time, place, event, result, number, duration, genealogy, or succession.
- NOT_ENOUGH_EVIDENCE: You do not have enough reliable information to determine whether the claim is correct or incorrect.

Instructions:
1. Examine every factual detail in the claim.
2. Choose SUPPORTED only when you can confirm the important details.
3. Choose REFUTED when you can identify at least one incorrect important detail.
4. Choose NOT_ENOUGH_EVIDENCE only when you cannot reliably confirm or refute the claim.
5. Do not guess.
6. Return exactly one valid JSON object and no additional text.

Claim:
{claim}

Output format:
{{"label":"SUPPORTED","explanation":"Briefly state the decisive fact."}}
"""

    if setting == "gold_evidence":
        evidence_text = format_gold_evidence(item)
        return f"""You are an evidence-based Vietnamese historical claim verifier.

Use ONLY the provided evidence. Do not use outside knowledge.

Your task is to compare the claim with the evidence and assign exactly one label.

Labels:
- SUPPORTED: The evidence explicitly states or clearly entails all main information in the claim.
- REFUTED: The evidence gives a different value for the same main information in the claim. This includes a different person, role/title, year/date, place, event result, number, duration, genealogy relation, or succession relation.
- NOT_ENOUGH_EVIDENCE: The evidence is related to the claim but does not contain enough information to confirm or contradict the claim.

Decision rules:
1. First identify the main factual detail in the claim.
2. Then find whether the evidence states the same detail.
3. If the evidence states the same detail with the same value, choose SUPPORTED.
4. If the evidence states the same detail with a different value, choose REFUTED.
5. If the evidence does not mention the detail needed to verify the claim, choose NOT_ENOUGH_EVIDENCE.
6. Absence of a detail is NOT a contradiction.
7. A different stated value is a contradiction.
8. Do not guess. Do not use outside historical knowledge.

Claim:
{claim}

Evidence:
{evidence_text}

Return only one JSON object:
{{"label":"SUPPORTED","explanation":"short reason based only on the evidence"}}
"""

    if setting == "retrieved_evidence":
        evidence_text, _ = format_retrieved_evidence(item)
        return f"""You are an evidence-based Vietnamese historical claim verifier.

You are given a claim and up to 5 retrieved evidence passages. Some passages may be irrelevant or noisy.

Use ONLY the retrieved evidence. Do not use outside knowledge.

Your task is to compare the claim with the relevant evidence passages and assign exactly one label.

Labels:
- SUPPORTED: At least one relevant evidence passage explicitly states or clearly entails all main information in the claim.
- REFUTED: At least one relevant evidence passage gives a different value for the same main information in the claim. This includes a different person, role/title, year/date, place, event result, number, duration, genealogy relation, or succession relation.
- NOT_ENOUGH_EVIDENCE: The retrieved evidence is missing, irrelevant, or related but does not contain enough information to confirm or contradict the claim.

Decision rules:
1. Ignore irrelevant evidence passages.
2. Do not use retrieval rank or score as proof.
3. First identify the main factual detail in the claim.
4. Then compare it with the most relevant evidence passage.
5. If a relevant passage states the same detail with the same value, choose SUPPORTED.
6. If a relevant passage states the same detail with a different value, choose REFUTED.
7. If no relevant passage contains enough information, choose NOT_ENOUGH_EVIDENCE.
8. Absence of a detail is NOT a contradiction.
9. A different stated value is a contradiction.
10. Do not guess. Do not use outside historical knowledge.

Claim:
{claim}

Retrieved evidence:
{evidence_text}

Return only one JSON object:
{{"label":"SUPPORTED","explanation":"short reason based only on the evidence"}}
"""

    raise ValueError(f"Unknown setting: {setting}")

In [7]:
def parse_verification_output(raw_text):
    text = (raw_text or "").strip()
    candidates = []

    # 1. JSON trong code block
    fence = re.search(
        r"```(?:json)?\s*(.*?)```",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )
    if fence:
        candidates.append(fence.group(1).strip())

    # 2. Toàn bộ text có thể là JSON
    candidates.append(text)

    # 3. JSON object nằm trong text
    json_obj = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if json_obj:
        candidates.append(json_obj.group(0).strip())

    last_label = ""
    last_explanation = ""

    for cand in candidates:
        try:
            obj = json.loads(cand)

            label = str(obj.get("label", "")).strip().upper()
            explanation = str(obj.get("explanation", "")).strip()

            last_label = label
            last_explanation = explanation

            if label not in LABELS:
                continue

            if not explanation:
                return label, "", "missing_explanation"

            return label, explanation, None

        except Exception:
            continue

    # Parse thất bại hoặc label không hợp lệ.
    # Vẫn trả về nhãn hợp lệ để pipeline không vỡ, nhưng đánh dấu parse_error.
    return (
        "NOT_ENOUGH_EVIDENCE",
        last_explanation,
        f"parse_error_or_invalid_label: {last_label}"
    )


def get_input_device(model):
    device_map = getattr(model, "hf_device_map", None)
    if isinstance(device_map, dict):
        for dev in device_map.values():
            if dev in ["cpu", "disk", "meta"]:
                continue
            if isinstance(dev, int):
                return torch.device(f"cuda:{dev}")
            if isinstance(dev, str) and dev.startswith("cuda"):
                return torch.device(dev)
    return next(model.parameters()).device


def render_chat(tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt


def generate_one(tokenizer, model, prompt):
    text = render_chat(tokenizer, prompt)
    input_device = get_input_device(model)

    # Không tự truncate evidence ở bước format. Đây chỉ là giới hạn an toàn cho tokenizer.
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=4096).to(input_device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            top_p=None, 
            top_k=None,
            temperature=None,
            num_beams=1,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## Load model

In [8]:
def get_quant_config(quantization):
    if quantization == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)
    if quantization == "4bit":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    return None


def load_llm(model_name, quantization):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=get_quant_config(quantization),
        device_map="auto",
        trust_remote_code=True,
        torch_dtype="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()
    return tokenizer, model


def cleanup_model(model=None, tokenizer=None):
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

## Chạy dự đoán theo từng model và setting

In [9]:
def run_one_setting(tokenizer, model, model_tag, model_name, quantization, setting):
    rows = []

    for item in tqdm(dataset, desc=f"{model_tag} | {setting}"):
        claim_id = str(item["claim_id"])
        prompt = build_prompt(item, setting)
        raw_response = generate_one(tokenizer, model, prompt)
        predicted_label, predicted_explanation, parse_error = (parse_verification_output(raw_response))

        retrieval_info = {
            "gold_evidence_ids": [str(x) for x in item["evidence_ids"]],
            "retrieved_chunk_ids": [],
            "retrieval_hit": None,
            "gold_rank": None,
            "missing_retrieval": False,
        }

        if setting == "retrieved_evidence":
            retrieval_info = get_retrieval_info(item)

        rows.append({
            "claim_id": claim_id,
            "claim": item["claim"],
            "gold_label": item["label"],
            "predicted_label": predicted_label,
            "gold_explanation": item["explanation"],
            "predicted_explanation": predicted_explanation,
            "model_tag": model_tag,
            "model_name": model_name,
            "quantization": quantization,
            "setting": setting,
            "raw_response": raw_response,
            "parse_error": bool(parse_error),
            "claim_type": item["claim_type"],
            "split": item["split"],
            "gold_chunk_id": str(item["chunk_id"]),
            "gold_evidence_ids": retrieval_info["gold_evidence_ids"],
            "retrieved_chunk_ids": retrieval_info["retrieved_chunk_ids"],
            "retrieval_hit": retrieval_info["retrieval_hit"],
            "gold_rank": retrieval_info["gold_rank"],
            "missing_retrieval": bool(retrieval_info["missing_retrieval"]),
        })

    return rows

In [10]:
all_predictions = []

for cfg in MODELS:
    print("\n" + "=" * 80)
    print("Model:", cfg)

    pending_settings = []
    for setting in SETTINGS:
        out_path = OUTPUT_DIR / f"{RUN_TAG}_{cfg['tag']}_{setting}_predictions.jsonl"

        if out_path.exists() and not FORCE_RERUN:
            print("Skip existing:", out_path)
            rows = load_jsonl(out_path)
            all_predictions.extend(rows)
        else:
            pending_settings.append(setting)

    if not pending_settings:
        continue

    tokenizer = None
    model = None

    try:
        tokenizer, model = load_llm(cfg["name"], cfg["quantization"])

        for setting in pending_settings:
            out_path = OUTPUT_DIR / f"{RUN_TAG}_{cfg['tag']}_{setting}_predictions.jsonl"
            rows = run_one_setting(
                tokenizer=tokenizer,
                model=model,
                model_tag=cfg["tag"],
                model_name=cfg["name"],
                quantization=cfg["quantization"],
                setting=setting,
            )
            save_jsonl(rows, out_path)
            print("Saved:", out_path)
            all_predictions.extend(rows)

    finally:
        print("Cleaning up model:", cfg["tag"])
        cleanup_model(model, tokenizer)

all_path = OUTPUT_DIR / f"{RUN_TAG}_verification_predictions_all.jsonl"
save_jsonl(all_predictions, all_path)
print("Saved all:", all_path)
print("Total predictions:", len(all_predictions))


Model: {'tag': 'qwen25_3b_8bit', 'name': 'Qwen/Qwen2.5-3B-Instruct', 'quantization': '8bit'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

qwen25_7b_8bit | claim_only:   0%|          | 0/264 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Cleaning up model: qwen25_7b_8bit


KeyboardInterrupt: 

## Metric tổng hợp

In [ ]:
def compute_metric_row(group):
    y_true = group["gold_label"].astype(str).tolist()
    y_pred = group["predicted_label"].astype(str).tolist()

    per_label = f1_score(y_true, y_pred, labels=LABELS, average=None, zero_division=0)

    return {
        "model_tag": group["model_tag"].iloc[0],
        "model_name": group["model_name"].iloc[0],
        "quantization": group["quantization"].iloc[0],
        "setting": group["setting"].iloc[0],
        "num_samples": len(group),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "f1_supported": per_label[0],
        "f1_refuted": per_label[1],
        "f1_not_enough_evidence": per_label[2],
        "parse_error_count": int(group["parse_error"].fillna(False).sum()),
        "missing_retrieval_count": int(group["missing_retrieval"].fillna(False).sum()),
    }


df = pd.DataFrame(all_predictions)
if df.empty:
    raise ValueError("No predictions found. Run prediction cells first.")

summary_rows = []
for _, group in df.groupby(["model_tag", "setting"], sort=False):
    summary_rows.append(compute_metric_row(group))

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / f"{RUN_TAG}_verification_summary.csv"
summary_df.to_csv(summary_path, index=False)
summary_df

## Confusion matrix

In [ ]:
cm_rows = []

for (model_tag, setting), group in df.groupby(["model_tag", "setting"], sort=False):
    cm = confusion_matrix(group["gold_label"], group["predicted_label"], labels=EXTENDED_LABELS)
    for i, true_label in enumerate(EXTENDED_LABELS):
        for j, pred_label in enumerate(EXTENDED_LABELS):
            cm_rows.append({
                "model_tag": model_tag,
                "setting": setting,
                "true_label": true_label,
                "predicted_label": pred_label,
                "count": int(cm[i, j]),
            })

cm_df = pd.DataFrame(cm_rows)
cm_path = OUTPUT_DIR / f"{RUN_TAG}_verification_confusion_matrix.csv"
cm_df.to_csv(cm_path, index=False)
cm_df.head(30)

## Metric theo claim_type

In [ ]:
claim_type_rows = []

for (model_tag, setting, claim_type), group in df.groupby(["model_tag", "setting", "claim_type"], sort=False):
    y_true = group["gold_label"].astype(str).tolist()
    y_pred = group["predicted_label"].astype(str).tolist()

    claim_type_rows.append({
        "model_tag": model_tag,
        "setting": setting,
        "claim_type": claim_type,
        "num_samples": len(group),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "parse_error_count": int(group["parse_error"].fillna(False).sum()),
    })

claim_type_df = pd.DataFrame(claim_type_rows)
claim_type_path = OUTPUT_DIR / f"{RUN_TAG}_verification_claim_type_metrics.csv"
claim_type_df.to_csv(claim_type_path, index=False)
claim_type_df.head(30)

## Phân tích retrieved_evidence theo retrieval_hit

Phần này chỉ áp dụng cho setting `retrieved_evidence`.

- `retrieval_hit = True`: top-5 retrieved evidence có chứa ít nhất một `gold_evidence_ids`.
- `gold_rank`: vị trí đầu tiên của gold evidence trong top-5.

Nếu `retrieval_hit=True` nhưng verifier sai, lỗi nghiêng về verifier/prompt/model. Nếu `retrieval_hit=False` và verifier sai, lỗi có thể bắt nguồn từ retrieval.

In [ ]:
retrieved_df = df[df["setting"] == "retrieved_evidence"].copy()

retrieval_hit_rows = []
if not retrieved_df.empty:
    for (model_tag, retrieval_hit), group in retrieved_df.groupby(["model_tag", "retrieval_hit"], dropna=False, sort=False):
        y_true = group["gold_label"].astype(str).tolist()
        y_pred = group["predicted_label"].astype(str).tolist()
        retrieval_hit_rows.append({
            "model_tag": model_tag,
            "retrieval_hit": retrieval_hit,
            "num_samples": len(group),
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
            "parse_error_count": int(group["parse_error"].fillna(False).sum()),
        })

retrieval_hit_df = pd.DataFrame(retrieval_hit_rows)
retrieval_hit_path = OUTPUT_DIR / f"{RUN_TAG}_verification_by_retrieval_hit.csv"
retrieval_hit_df.to_csv(retrieval_hit_path, index=False)
retrieval_hit_df

In [ ]:
# File chẩn đoán theo từng claim cho setting retrieved_evidence.
retrieval_diag_cols = [
    "model_tag", "claim_id", "gold_label", "predicted_label", "claim_type",
    "gold_evidence_ids", "retrieved_chunk_ids", "retrieval_hit", "gold_rank", "parse_error"
]

retrieval_diag_df = retrieved_df[retrieval_diag_cols].copy() if not retrieved_df.empty else pd.DataFrame(columns=retrieval_diag_cols)
retrieval_diag_path = OUTPUT_DIR / f"{RUN_TAG}_verification_retrieval_diagnostics.csv"
retrieval_diag_df.to_csv(retrieval_diag_path, index=False)
retrieval_diag_df.head(20)

## Nén output

In [ ]:
zip_base = Path(f"/kaggle/working/verification_outputs_clean_{RUN_TAG}")
zip_path = shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR)

print("ZIP:", zip_path)
print("Files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p.name)